In [1]:
"""
DPad Button — 3D Geometry Viewer
==================================
Run this script in Google Colab (or any Jupyter/IPython environment).
Generates the DPad mesh, exports a binary STL, and opens an interactive
Plotly 3D viewer + a 4-panel multi-view (Top / Front / Side / Isometric).

Zero pip installs needed — numpy and plotly are pre-installed in Colab.

Leg length: 9 mm  (modified from original 18 mm, scaled ×0.5)

Usage in Colab:
    1. Upload this file to Colab
    2. Run:  !python dpad_3d_viewer.py
       OR paste into a cell and run directly.
"""

# ══════════════════════════════════════════════════════════════════════ #
#  Imports                                                               #
# ══════════════════════════════════════════════════════════════════════ #

import math
import struct
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ══════════════════════════════════════════════════════════════════════ #
#  Dimensions  (edit any value to change the model)                     #
# ══════════════════════════════════════════════════════════════════════ #

# ── Plus-Cross Body ───────────────────────────────────────────────────
LEG_LENGTH =   9.0   # mm  ← MODIFIED  (original: 18 mm, scaled ×0.5)
LEG_HEIGHT =   3.0   # mm  — vertical thickness of the body
LEG_WIDTH  =   6.0   # mm  — width of each arm
NUM_LEGS   =   4
HW         = LEG_WIDTH / 2.0      # half-width = 3.0 mm

# ── Cylindrical Alignment Posts  (bottom face, one per leg) ──────────
POST_RADIUS               = 1.0   # mm  (assumed; not explicitly stated)
POST_HEIGHT               = 1.2   # mm
POST_FILLET_RADIUS        = 0.25  # mm  (top-edge fillet, noted in dims)
POST_INSET_EXTERNAL_EDGE  = 2.45  # mm  — inward from outer leg edge
POST_OFFSET_FROM_LEG_AXIS = 7.0   # mm  — along the leg's long axis
NUM_POSTS = 4

# ── Top Circular Disk  (thumb-contact cap) ───────────────────────────
DISK_RADIUS = 6.5    # mm
DISK_HEIGHT = 1.0    # mm

# ── Mesh quality ─────────────────────────────────────────────────────
SEGMENTS    = 48     # radial segments for cylinders
                     # 16 = blocky/fast   32 = balanced   64 = smooth

# ── Output file ──────────────────────────────────────────────────────
STL_FILE = "dpad_button_9mm.stl"


# ══════════════════════════════════════════════════════════════════════ #
#  Mesh helper functions                                                 #
# ══════════════════════════════════════════════════════════════════════ #

def tri_normal(v0, v1, v2):
    """Unit outward normal for a triangle (CCW winding = outward)."""
    n = np.cross(v1 - v0, v2 - v0)
    length = np.linalg.norm(n)
    return n / length if length > 1e-12 else np.array([0.0, 1.0, 0.0])


def box_mesh(xmin, xmax, ymin, ymax, zmin, zmax):
    """
    12 triangles for an axis-aligned box with outward CCW winding.
    Returns list of (v0, v1, v2) numpy arrays.
    """
    lbf = np.array([xmin, ymin, zmin]); rbf = np.array([xmax, ymin, zmin])
    ltf = np.array([xmin, ymax, zmin]); rtf = np.array([xmax, ymax, zmin])
    lbb = np.array([xmin, ymin, zmax]); rbb = np.array([xmax, ymin, zmax])
    ltb = np.array([xmin, ymax, zmax]); rtb = np.array([xmax, ymax, zmax])
    return [
        (lbf, rbb, rbf), (lbf, lbb, rbb),   # bottom  (normal -Y)
        (ltf, rtf, rtb), (ltf, rtb, ltb),   # top     (normal +Y)
        (lbf, rbf, rtf), (lbf, rtf, ltf),   # front   (normal -Z)
        (rbb, lbb, ltb), (rbb, ltb, rtb),   # back    (normal +Z)
        (lbb, lbf, ltf), (lbb, ltf, ltb),   # left    (normal -X)
        (rbf, rbb, rtb), (rbf, rtb, rtf),   # right   (normal +X)
    ]


def cylinder_mesh(cx, cy, cz, radius, height, segments=32,
                  cap_top=True, cap_bottom=True):
    """
    Triangle soup for a vertical cylinder.
    Bottom centre = (cx, cy, cz).  Top centre = (cx, cy+height, cz).
    Returns list of (v0, v1, v2).
    """
    tris   = []
    angles = [2 * math.pi * i / segments for i in range(segments)]
    bot = [np.array([cx + radius*math.cos(a), cy,          cz + radius*math.sin(a)]) for a in angles]
    top = [np.array([cx + radius*math.cos(a), cy + height, cz + radius*math.sin(a)]) for a in angles]
    bc  = np.array([cx, cy,          cz])
    tc  = np.array([cx, cy + height, cz])
    for i in range(segments):
        j = (i + 1) % segments
        tris += [(bot[i], top[i], top[j]), (bot[i], top[j], bot[j])]
        if cap_bottom: tris.append((bc, bot[j], bot[i]))
        if cap_top:    tris.append((tc, top[i], top[j]))
    return tris


# ══════════════════════════════════════════════════════════════════════ #
#  Build DPad geometry                                                   #
# ══════════════════════════════════════════════════════════════════════ #

def build_triangles(segments=SEGMENTS):
    """
    Assemble the full DPad triangle soup.

    Layer diagram (Y axis, side view):
      ┌────────────────┐  Y = LEG_HEIGHT + DISK_HEIGHT  (top of disk)
      │   top disk     │
      ├────────────────┤  Y = LEG_HEIGHT                (top of body)
      │                │
      │   plus body    │
      │                │
      ├────────────────┤  Y = 0                         (bottom face)
      │  ● post ● post │  (posts hang below)
      └────────────────┘  Y = -POST_HEIGHT

    Top view (XZ plane):
            ┌──────┐
            │  N   │   ← LEG_WIDTH = 6 mm
       ┌────┼──────┼────┐
       │ W  │ hub  │ E  │   LEG_LENGTH = 9 mm each arm
       └────┼──────┼────┘
            │  S   │
            └──────┘
    """
    tris = []
    L, H = LEG_LENGTH, LEG_HEIGHT

    # 1. Plus-cross body — 5 non-overlapping axis-aligned boxes
    for b in [
        (-HW, +HW, 0, H, -HW, +HW),   # central hub
        (-HW, +HW, 0, H,  -L, -HW),   # arm North (-Z)
        (-HW, +HW, 0, H, +HW,  +L),   # arm South (+Z)
        ( -L, -HW, 0, H, -HW, +HW),   # arm West  (-X)
        (+HW,  +L, 0, H, -HW, +HW),   # arm East  (+X)
    ]:
        tris.extend(box_mesh(*b))

    # 2. Top disk — centred on hub, sits on top face
    tris.extend(
        cylinder_mesh(0.0, H, 0.0, DISK_RADIUS, DISK_HEIGHT, segments=segments)
    )

    # 3. Bottom alignment posts — one per leg, hang below bottom face
    OFF = POST_OFFSET_FROM_LEG_AXIS
    for (px, pz) in [(OFF, 0.0), (-OFF, 0.0), (0.0, OFF), (0.0, -OFF)]:
        tris.extend(
            cylinder_mesh(px, -POST_HEIGHT, pz, POST_RADIUS, POST_HEIGHT,
                          segments=segments)
        )
    return tris


# ══════════════════════════════════════════════════════════════════════ #
#  Binary STL writer                                                     #
# ══════════════════════════════════════════════════════════════════════ #

def write_binary_stl(triangles, filepath):
    """Write triangle list to a binary STL file (no external libraries)."""
    n      = len(triangles)
    header = b"DPad Button 9mm leg - procedural geometry".ljust(80, b"\x00")
    with open(filepath, "wb") as f:
        f.write(header)
        f.write(struct.pack("<I", n))
        for (v0, v1, v2) in triangles:
            v0  = np.asarray(v0,  dtype=np.float32)
            v1  = np.asarray(v1,  dtype=np.float32)
            v2  = np.asarray(v2,  dtype=np.float32)
            nrm = tri_normal(v0, v1, v2).astype(np.float32)
            f.write(struct.pack("<3f", *nrm))
            f.write(struct.pack("<3f", *v0))
            f.write(struct.pack("<3f", *v1))
            f.write(struct.pack("<3f", *v2))
            f.write(struct.pack("<H",   0))
    return n


# ══════════════════════════════════════════════════════════════════════ #
#  Convert triangle soup → Plotly indexed mesh                          #
# ══════════════════════════════════════════════════════════════════════ #

def tris_to_plotly(triangles):
    """
    Deduplicate vertices and build face index arrays for Plotly Mesh3d.
    Returns: x, y, z (vertex coords), fi, fj, fk (face indices).
    """
    vertex_map = {}
    verts      = []

    def vid(v):
        key = (round(float(v[0]), 6),
               round(float(v[1]), 6),
               round(float(v[2]), 6))
        if key not in vertex_map:
            vertex_map[key] = len(verts)
            verts.append(key)
        return vertex_map[key]

    fi, fj, fk = [], [], []
    for (v0, v1, v2) in triangles:
        fi.append(vid(v0)); fj.append(vid(v1)); fk.append(vid(v2))

    v = np.array(verts)
    return v[:, 0], v[:, 1], v[:, 2], fi, fj, fk


# ══════════════════════════════════════════════════════════════════════ #
#  Interactive 3D viewer                                                 #
# ══════════════════════════════════════════════════════════════════════ #

def show_dpad_3d(triangles):
    """
    Full interactive Plotly Mesh3d viewer.
    Controls: Left-drag=rotate | Right-drag=pan | Scroll=zoom | Dbl-click=reset
    """
    print("  Building Plotly index …")
    x, y, z, fi, fj, fk = tris_to_plotly(triangles)
    print(f"  Unique vertices : {len(x):,}  |  Faces : {len(fi):,}")

    fig = go.Figure()

    # Main DPad mesh coloured by height (Y)
    fig.add_trace(go.Mesh3d(
        x=x, y=y, z=z,
        i=fi, j=fj, k=fk,
        intensity=y,
        colorscale=[
            [0.00, "#0a0a1a"],
            [0.20, "#0d2137"],
            [0.50, "#0e4d70"],
            [0.75, "#4a8fa8"],
            [1.00, "#e94560"],
        ],
        showscale=True,
        colorbar=dict(
            title=dict(text="Height Y (mm)", side="right",
                       font=dict(color="#cccccc", size=12)),
            tickfont=dict(color="#cccccc"),
            thickness=14, len=0.65,
            bgcolor="rgba(0,0,0,0)",
            bordercolor="#333333",
        ),
        flatshading=False,
        lighting=dict(
            ambient=0.30, diffuse=0.90,
            specular=0.35, roughness=0.35, fresnel=0.15,
        ),
        lightposition=dict(x=300, y=500, z=400),
        hovertemplate=(
            "X: %{x:.2f} mm<br>"
            "Y: %{y:.2f} mm<br>"
            "Z: %{z:.2f} mm<extra></extra>"
        ),
        name="DPad",
    ))

    # Reference grid lines at Y=0 (bottom face level)
    L = LEG_LENGTH * 1.08
    for xs, ys, zs in [
        ([-L, L], [0, 0], [0, 0]),
        ([0,  0], [0, 0], [-L, L]),
    ]:
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs, mode="lines",
            line=dict(color="#333355", width=1),
            showlegend=False, hoverinfo="skip",
        ))

    # 3D callout annotations
    scene_annotations = [
        dict(x=LEG_LENGTH, y=LEG_HEIGHT / 2, z=0,
             text=f"<b>Leg tip</b><br>{LEG_LENGTH:.0f} mm",
             showarrow=True, arrowcolor="#aaaaaa",
             font=dict(color="#ffffff", size=11)),
        dict(x=0, y=LEG_HEIGHT + DISK_HEIGHT, z=0,
             text=f"<b>Top disk</b><br>r = {DISK_RADIUS} mm",
             showarrow=True, arrowcolor="#e94560",
             font=dict(color="#e94560", size=11)),
        dict(x=POST_OFFSET_FROM_LEG_AXIS, y=-POST_HEIGHT / 2, z=0,
             text=f"<b>Post</b><br>h = {POST_HEIGHT} mm",
             showarrow=True, arrowcolor="#4a8fa8",
             font=dict(color="#4a8fa8", size=11)),
    ]

    fig.update_layout(
        title=dict(
            text=(
                "<b>DPad Button — Interactive 3D View</b><br>"
                f"<sup>Leg {LEG_LENGTH}×{LEG_WIDTH}×{LEG_HEIGHT} mm  |  "
                f"Disk r={DISK_RADIUS} mm  |  Posts h={POST_HEIGHT} mm  |  "
                f"{len(triangles):,} triangles</sup>"
            ),
            x=0.5, xanchor="center",
            font=dict(size=16, color="#ffffff"),
        ),
        paper_bgcolor="#0a0a0f",
        scene=dict(
            bgcolor="#0a0a0f",
            xaxis=dict(title="X (mm)", gridcolor="#1e1e2e",
                       showbackground=True, backgroundcolor="#0f0f1a",
                       tickfont=dict(color="#888888"),
                       titlefont=dict(color="#aaaaaa"),
                       zeroline=True, zerolinecolor="#333355"),
            yaxis=dict(title="Y — Height (mm)", gridcolor="#1e1e2e",
                       showbackground=True, backgroundcolor="#0f0f1a",
                       tickfont=dict(color="#888888"),
                       titlefont=dict(color="#aaaaaa"),
                       zeroline=True, zerolinecolor="#333355"),
            zaxis=dict(title="Z (mm)", gridcolor="#1e1e2e",
                       showbackground=True, backgroundcolor="#0f0f1a",
                       tickfont=dict(color="#888888"),
                       titlefont=dict(color="#aaaaaa"),
                       zeroline=True, zerolinecolor="#333355"),
            aspectmode="data",
            camera=dict(eye=dict(x=1.6, y=1.2, z=1.6),
                        up=dict(x=0, y=1, z=0)),
            annotations=scene_annotations,
        ),
        margin=dict(l=0, r=0, t=90, b=0),
        height=720,
        annotations=[dict(
            text="🖱 Left-drag: rotate  |  Right-drag: pan  |  Scroll: zoom  |  Double-click: reset",
            showarrow=False, xref="paper", yref="paper",
            x=0.5, y=-0.01, xanchor="center",
            font=dict(size=11, color="#555555"),
        )],
    )

    fig.show()


# ══════════════════════════════════════════════════════════════════════ #
#  4-Panel multi-view                                                    #
# ══════════════════════════════════════════════════════════════════════ #

def show_multiview(triangles):
    """
    2×2 panel: Top view / Front view / Side view / Isometric.
    Each panel is independently rotatable.
    """
    print("  Building multi-view panel …")
    x, y, z, fi, fj, fk = tris_to_plotly(triangles)

    cameras = [
        ("Top View (XZ)",   dict(eye=dict(x=0,   y=6,   z=0),   up=dict(x=0, y=0, z=1))),
        ("Front View (XY)", dict(eye=dict(x=0,   y=0,   z=5),   up=dict(x=0, y=1, z=0))),
        ("Side View (ZY)",  dict(eye=dict(x=5,   y=0,   z=0),   up=dict(x=0, y=1, z=0))),
        ("Isometric",       dict(eye=dict(x=1.8, y=1.4, z=1.8), up=dict(x=0, y=1, z=0))),
    ]

    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}],
               [{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=[c[0] for c in cameras],
        horizontal_spacing=0.02,
        vertical_spacing=0.06,
    )

    mesh_kw = dict(
        x=x, y=y, z=z, i=fi, j=fj, k=fk,
        intensity=y,
        colorscale=[
            [0.00, "#0a0a1a"], [0.20, "#0d2137"],
            [0.50, "#0e4d70"], [0.75, "#4a8fa8"],
            [1.00, "#e94560"],
        ],
        showscale=False, flatshading=True,
        lighting=dict(ambient=0.4, diffuse=0.8, specular=0.2, roughness=0.5),
        hoverinfo="skip", showlegend=False,
    )

    scene_ids = ["scene", "scene2", "scene3", "scene4"]
    positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

    for (row, col), (label, cam), sid in zip(positions, cameras, scene_ids):
        fig.add_trace(go.Mesh3d(**mesh_kw, name=label), row=row, col=col)
        fig.update_layout(**{sid: dict(
            bgcolor="#0f0f1a",
            camera=cam,
            aspectmode="data",
            xaxis=dict(showgrid=True, gridcolor="#1e1e2e",
                       showticklabels=False,
                       showbackground=True, backgroundcolor="#0a0a12"),
            yaxis=dict(showgrid=True, gridcolor="#1e1e2e",
                       showticklabels=False,
                       showbackground=True, backgroundcolor="#0a0a12"),
            zaxis=dict(showgrid=True, gridcolor="#1e1e2e",
                       showticklabels=False,
                       showbackground=True, backgroundcolor="#0a0a12"),
        )})

    fig.update_layout(
        title=dict(
            text="<b>DPad Button — Multi-View Panel</b>",
            x=0.5, xanchor="center",
            font=dict(size=15, color="#ffffff"),
        ),
        paper_bgcolor="#07070f",
        height=820,
        margin=dict(l=10, r=10, t=65, b=10),
    )
    fig.show()


# ══════════════════════════════════════════════════════════════════════ #
#  Geometry report                                                       #
# ══════════════════════════════════════════════════════════════════════ #

def print_report(triangles):
    hub_area        = LEG_WIDTH ** 2
    leg_plan        = (LEG_LENGTH - HW) * LEG_WIDTH
    total_plan      = hub_area + NUM_LEGS * leg_plan
    body_vol        = total_plan * LEG_HEIGHT
    disk_vol        = math.pi * DISK_RADIUS**2 * DISK_HEIGHT
    post_vol        = math.pi * POST_RADIUS**2 * POST_HEIGHT * NUM_POSTS

    all_v = np.array([v for t in triangles for v in t])
    lo, hi = all_v.min(axis=0), all_v.max(axis=0)

    print("=" * 58)
    print("  DPad Button — Geometry Report")
    print("=" * 58)
    print()
    print("── Plus-Cross Body ───────────────────────────────────")
    print(f"  Leg length          : {LEG_LENGTH:.1f} mm  ← MODIFIED (orig 18 mm, ×{LEG_LENGTH/18:.2f})")
    print(f"  Leg height          : {LEG_HEIGHT:.1f} mm")
    print(f"  Leg width           : {LEG_WIDTH:.1f} mm")
    print(f"  Total span (tip↔tip): {2*LEG_LENGTH:.0f} mm")
    print(f"  Body plan area      : {total_plan:.2f} mm²")
    print(f"  Body volume (approx): {body_vol:.2f} mm³")
    print()
    print("── Cylindrical Posts ─────────────────────────────────")
    print(f"  Count               : {NUM_POSTS}")
    print(f"  Radius              : {POST_RADIUS:.2f} mm  (assumed)")
    print(f"  Height              : {POST_HEIGHT:.2f} mm")
    print(f"  Top-edge fillet     : {POST_FILLET_RADIUS:.2f} mm")
    print(f"  Inset from edge     : {POST_INSET_EXTERNAL_EDGE:.2f} mm")
    print(f"  Offset along leg    : {POST_OFFSET_FROM_LEG_AXIS:.2f} mm")
    print()
    print("── Top Circular Disk ─────────────────────────────────")
    print(f"  Radius              : {DISK_RADIUS:.2f} mm")
    print(f"  Height              : {DISK_HEIGHT:.2f} mm")
    print(f"  Volume              : {disk_vol:.4f} mm³")
    print()
    print("── Mesh Stats ────────────────────────────────────────")
    print(f"  Triangles           : {len(triangles):,}")
    print(f"  Cylinder segments   : {SEGMENTS}")
    print()
    print("── Actual Bounding Box ───────────────────────────────")
    for ax, i in zip(["X", "Y", "Z"], range(3)):
        print(f"  {ax}: {lo[i]:9.3f} … {hi[i]:9.3f} mm  (span {hi[i]-lo[i]:.3f} mm)")
    print("=" * 58)


# ══════════════════════════════════════════════════════════════════════ #
#  Main                                                                  #
# ══════════════════════════════════════════════════════════════════════ #

if __name__ == "__main__":

    # ── 1. Build mesh ─────────────────────────────────────────────────
    print(f"\nBuilding DPad mesh  (LEG_LENGTH={LEG_LENGTH} mm, segments={SEGMENTS}) …")
    triangles = build_triangles(segments=SEGMENTS)

    # ── 2. Print geometry report ──────────────────────────────────────
    print_report(triangles)

    # ── 3. Write binary STL ───────────────────────────────────────────
    print(f"\nWriting STL → '{STL_FILE}' …")
    n       = write_binary_stl(triangles, STL_FILE)
    size_kb = (80 + 4 + n * 50) / 1024
    print(f"  ✓  {n:,} triangles  |  {size_kb:.1f} KB")

    # ── 4. Interactive 3D viewer ──────────────────────────────────────
    print("\nRendering interactive 3D view …")
    show_dpad_3d(triangles)

    # ── 5. Multi-view panel ───────────────────────────────────────────
    print("\nRendering multi-view panel …")
    show_multiview(triangles)

    # ── 6. Auto-download STL in Colab ─────────────────────────────────
    try:
        from google.colab import files
        print(f"\nDownloading '{STL_FILE}' …")
        files.download(STL_FILE)
        print("✅ Download triggered.")
    except ImportError:
        print(f"\nℹ  Not running in Colab — STL saved locally as '{STL_FILE}'")


Building DPad mesh  (LEG_LENGTH=9.0 mm, segments=48) …
  DPad Button — Geometry Report

── Plus-Cross Body ───────────────────────────────────
  Leg length          : 9.0 mm  ← MODIFIED (orig 18 mm, ×0.50)
  Leg height          : 3.0 mm
  Leg width           : 6.0 mm
  Total span (tip↔tip): 18 mm
  Body plan area      : 180.00 mm²
  Body volume (approx): 540.00 mm³

── Cylindrical Posts ─────────────────────────────────
  Count               : 4
  Radius              : 1.00 mm  (assumed)
  Height              : 1.20 mm
  Top-edge fillet     : 0.25 mm
  Inset from edge     : 2.45 mm
  Offset along leg    : 7.00 mm

── Top Circular Disk ─────────────────────────────────
  Radius              : 6.50 mm
  Height              : 1.00 mm
  Volume              : 132.7323 mm³

── Mesh Stats ────────────────────────────────────────
  Triangles           : 1,020
  Cylinder segments   : 48

── Actual Bounding Box ───────────────────────────────
  X:    -9.000 …     9.000 mm  (span 18.000 mm)
  Y:


Rendering multi-view panel …
  Building multi-view panel …


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download triggered.
